In [ ]:
import os, re, random, time, torch

cache_path = "/workspace/user/chainsearch_archives/data_pts/exp27_n3_countdown_data_mode-random_backtrack_NALL_S3-3_Tfromtrace_rb10_rpn2500_compact0_seed42_fmtv2.pt"
split = "train"
preview_path = "dataset_preview.txt"
seed_split = 42
seed_preview = 123
seed_file = 777
per_n_preview = 10
per_n_file = 50
width = 80

# ---- Load ----
data = torch.load(cache_path, map_location="cpu")  # if you ever hit weights_only issues: add weights_only=False
ins  = data.get("prompts") or data.get("ins")
outs = data.get("traces")  or data.get("outs")
assert ins is not None and outs is not None, f"Unexpected keys: {list(data.keys())}"
assert len(ins) == len(outs)
print(f"loaded: {len(ins):,} pairs")

def parse_n(prompt_text: str) -> int:
    m = re.search(r"IN:\s*\[\s*([^\]]*?)\s*\]", prompt_text)
    if not m: return -1
    return len(re.findall(r"-?\d+", m.group(1)))

def count_backs(trace: str) -> int:
    return len(re.findall(r"\bBACK\b", trace))

def group_by_n(indices):
    g = {}
    for k in indices:
        n = parse_n(ins[k])
        g.setdefault(n, []).append(k)
    return g

def header(title: str, ch="="):
    bar = ch * width
    return f"\n{bar}\n{title.center(width)}\n{bar}\n"

# def pretty_example(tag, idx):
#     inp, out = ins[idx], outs[idx]
#     n = parse_n(inp)
#     full = f"{inp} SEP {out}"   # build only when needed
#     line = "-" * width
#     return (
#         f"{tag}\n"
#         f"n: {n}\n"
#         f"Backtracks: {count_backs(out)}\n"
#         f"Input:   {inp}\n"
#         f"Target:  {out}\n"
#         f"Full:    {full}\n{line}\n"
#     )

def pretty_trace(s: str) -> str:
    # put each semicolon-delimited segment on its own line
    # keeps the semicolon at the end of each line
    return ";\n".join(part.strip() for part in s.split(";") if part.strip())

def pretty_full(s: str) -> str:
    # same idea, but keeps " SEP " readable too
    if " SEP " not in s:
        return pretty_trace(s)
    left, right = s.split(" SEP ", 1)
    return left.strip() + "\nSEP\n" + pretty_trace(right)

def pretty_example(tag, idx):
    inp = ins[idx]
    out = outs[idx]
    n = parse_n(inp)
    full = f"{inp} SEP {out}"
    line = "-" * width
    return (
        f"{tag}\n"
        f"n: {n}\n"
        f"Backtracks: {count_backs(out)}\n"
        f"Input:   {inp}\n"
        f"Target:\n{pretty_trace(out)}\n"
        f"Full:\n{pretty_full(full)}\n{line}\n"
    )


# ---- Deterministic 90/10 split (index-only, no copying strings) ----
idx = list(range(len(ins)))
random.Random(seed_split).shuffle(idx)
cut = int(len(idx) * 0.9)
view_idx = idx[:cut] if split == "train" else idx[cut:]

grp = group_by_n(view_idx)
ns_sorted = sorted([k for k in grp.keys() if k >= 0])

print(header(f"Dataset Preview by n (split={split})", ch="="))
rng = random.Random(seed_preview)
for n in ns_sorted:
    ids = grp[n]
    k = min(per_n_preview, len(ids))
    chosen = rng.sample(ids, k)
    print(header(f"n = {n}  |  showing {k} samples", ch="-"), end="")
    for j, ex_idx in enumerate(chosen, 1):
        print(pretty_example(f"({j:02d})", ex_idx), end="")

ts = time.strftime("%Y-%m-%d %H:%M:%S")
with open(preview_path, "a", encoding="utf-8") as f:
    f.write(header(f"Countdown Dataset Dump | {ts} | split={split} | path={cache_path}", ch="="))
    rng2 = random.Random(seed_file)
    for n in ns_sorted:
        ids = grp[n]
        k = min(per_n_file, len(ids))
        chosen = rng2.sample(ids, k)
        f.write(header(f"n = {n}  |  writing {k} samples", ch="-"))
        for j, ex_idx in enumerate(chosen, 1):
            f.write(pretty_example(f"({j:02d})", ex_idx))

print(f"\nAppended per-n dumps to: {preview_path}")


loaded: 1,114,775 pairs

                       Dataset Preview by n (split=train)                       


--------------------------------------------------------------------------------
                          n = 3  |  showing 10 samples                          
--------------------------------------------------------------------------------
(01)
n: 3
Backtracks: 3
Input:   IN: [ 1, 1, 9 ] TGT: 8
Target:
[ 1, 1, 9 ] -> 1 + 1 = 2 -> [ 2, 9 ];
[ 2, 9 ] -> 2 + 9 = 11 -> [ 11 ];
BACK -> [ 1, 1, 9 ];
[ 1, 1, 9 ] -> 9 - 1 = 8 -> [ 1, 8 ];
[ 1, 8 ] -> 8 / 1 = 8 -> [ 8 ];
BACK -> [ 1, 1, 9 ];
[ 1, 1, 9 ] -> 9 / 1 = 9 -> [ 1, 9 ];
BACK -> [ 1, 1, 9 ];
[ 1, 1, 9 ] -> 9 - 1 = 8 -> [ 1, 8 ];
[ 1, 8 ] -> 8 / 1 = 8 -> [ 8 ];
SUCCESS;
EXPR (((9 - 1) / 1))
Full:
IN: [ 1, 1, 9 ] TGT: 8
SEP
[ 1, 1, 9 ] -> 1 + 1 = 2 -> [ 2, 9 ];
[ 2, 9 ] -> 2 + 9 = 11 -> [ 11 ];
BACK -> [ 1, 1, 9 ];
[ 1, 1, 9 ] -> 9 - 1 = 8 -> [ 1, 8 ];
[ 1, 8 ] -> 8 / 1 = 8 -> [ 8 ];
BACK -> [ 1, 1, 9 ];
[ 1, 1, 9 ] -> 9 / 1 = 9 